In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# --- Same split as the simple run above (train on last 4 seasons, test on 25-26),
#     but using predict_proba() to get real probabilities instead of a hard label,
#     converted to odds (1/p) and compared against the bookmakers' average odds. ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons_with_bookies.csv")

train_seasons = ["21-22", "22-23", "23-24", "24-25"]
test_season = "25-26"

train_df = all_df[all_df["Season"].isin(train_seasons)]
test_df = all_df[all_df["Season"] == test_season].copy()

# Exclude the usual identifiers/leakage columns AND the bookmaker odds columns
# themselves -- those are what we're comparing against, not features to train on.
drop_cols = [
    "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season",
    "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds", "NumBookies"
]

X_train = train_df.drop(columns=drop_cols).fillna(0)
y_train = train_df["FTR"]

X_test = test_df.drop(columns=drop_cols).fillna(0)

print(f"Train: {train_seasons} -> {X_train.shape[0]} matches, {X_train.shape[1]} features")
print(f"Test: {test_season} -> {X_test.shape[0]} matches")

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf.fit(X_train, y_train)

# --- Convert predicted probabilities into "model odds" ---
proba = rf.predict_proba(X_test)
model_odds = 1 / np.clip(proba, 1e-6, None)  # clip guards against a divide-by-zero on a 0% class

class_to_odds_col = {"H": "ModelHomeOdds", "D": "ModelDrawOdds", "A": "ModelAwayOdds"}
odds_cols_ordered = [class_to_odds_col[c] for c in rf.classes_]
model_odds_df = pd.DataFrame(model_odds, columns=odds_cols_ordered, index=test_df.index)

comparison = pd.concat([
    test_df[["Date", "HomeTeam", "AwayTeam", "FTR", "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds"]],
    model_odds_df[["ModelHomeOdds", "ModelDrawOdds", "ModelAwayOdds"]]
], axis=1)

# The market's odds carry a built-in margin (overround), but the model's
# probabilities sum to exactly 1 -- de-vig the market odds too so the
# comparison isn't just "model odds are bigger because they have no margin".
overround = 1 / comparison["AvgHomeOdds"] + 1 / comparison["AvgDrawOdds"] + 1 / comparison["AvgAwayOdds"]
comparison["FairMarketHomeOdds"] = 1 / (1 / comparison["AvgHomeOdds"] / overround)
comparison["FairMarketDrawOdds"] = 1 / (1 / comparison["AvgDrawOdds"] / overround)
comparison["FairMarketAwayOdds"] = 1 / (1 / comparison["AvgAwayOdds"] / overround)

pd.set_option("display.width", 160)
print("\nSample comparison (first 10 matches):")
print(comparison.head(10).round(2))

print("\n=== Model odds vs. market odds ===")
for outcome, model_col, market_col, fair_col in [
    ("Home", "ModelHomeOdds", "AvgHomeOdds", "FairMarketHomeOdds"),
    ("Draw", "ModelDrawOdds", "AvgDrawOdds", "FairMarketDrawOdds"),
    ("Away", "ModelAwayOdds", "AvgAwayOdds", "FairMarketAwayOdds"),
]:
    mean_model = comparison[model_col].mean()
    mean_market = comparison[market_col].mean()
    mean_fair = comparison[fair_col].mean()
    mae_vs_market = (comparison[model_col] - comparison[market_col]).abs().mean()
    mae_vs_fair = (comparison[model_col] - comparison[fair_col]).abs().mean()
    corr = comparison[model_col].corr(comparison[market_col])
    print(
        f"{outcome:5s}: mean model={mean_model:5.2f}  mean market={mean_market:5.2f}  "
        f"mean fair-market={mean_fair:5.2f}  MAE(vs market)={mae_vs_market:.2f}  "
        f"MAE(vs fair)={mae_vs_fair:.2f}  corr={corr:.3f}"
    )



Train: ['21-22', '22-23', '23-24', '24-25'] -> 1439 matches, 41 features
Test: 25-26 -> 360 matches

Sample comparison (first 10 matches):
            Date       HomeTeam        AwayTeam FTR  AvgHomeOdds  AvgDrawOdds  AvgAwayOdds  ModelHomeOdds  ModelDrawOdds  ModelAwayOdds  FairMarketHomeOdds  \
5033  2025-08-30        Chelsea          Fulham   H         1.55         4.39         5.60           1.90           3.88           4.65                1.63   
5034  2025-08-30     Man United         Burnley   H         1.35         5.26         8.34           2.15           3.68           3.81                1.42   
5035  2025-08-30     Sunderland       Brentford   H         2.94         3.29         2.47           4.38           3.46           2.07                3.08   
5036  2025-08-30      Tottenham     Bournemouth   A         1.74         4.04         4.35           2.46           3.99           2.92                1.83   
5037  2025-08-30         Wolves         Everton   A         2.67  